In [41]:
LabelToID = {
    'O': 0,
    'B-Date': 1,
    'I-Date': 2,
    'B-Person': 3,
    'I-Person': 4,
    'B-Location': 5,
    'I-Location': 6,
    'B-Facility': 7,
    'I-Facility': 8,
    'B-Organization': 9,
    'I-Organization': 10,
    'B-Misc': 11,
    'I-Misc': 12,
    'B-Money': 13,
    'I-Money': 14,
    'B-NORP': 15,
    'I-NORP': 16,
    'B-Product': 17,
    'I-Product': 18}

In [42]:
import os
import glob
import pandas as pd

def load_tsvs_from_folder(folder_path):
    all_sentences = []
    all_ner_tags = []
    all_regions = []

    search_pattern = os.path.join(folder_path, "*.tsv")
    file_list = glob.glob(search_pattern)
    
    for file_path in file_list:
        filename = os.path.basename(file_path)
        region_name = filename.split('_')[0]
        current_sentence = []
        current_tags = []

        with open(file_path, 'r', encoding='utf-8') as f:
            for _ in range(2):       # Skip the first two lines
                try:
                    next(f)
                except StopIteration:
                    break

            for line in f:
                line = line.strip()

                if not line:    # EMPTY LINE: Sentence is over
                    if current_sentence:
                        all_sentences.append(list(current_sentence))
                        all_ner_tags.append(list(current_tags))
                        all_regions.append(region_name)
                        current_sentence.clear()
                        current_tags.clear()
                    continue 
                    
                parts = line.split('\t')    # TEXT: Extract token and primary tag
                if len(parts) >= 2:
                    token = parts[0].strip()
                    raw_tag = parts[1].strip() # Grab the FIRST tag to avoid nested tags

                    current_sentence.append(token)
                    current_tags.append(raw_tag)

            if current_sentence:        # END OF FILE: Catch the final sentence
                all_sentences.append(list(current_sentence))
                all_ner_tags.append(list(current_tags))
                all_regions.append(region_name)

    df = pd.DataFrame({'tokens': all_sentences, 'ner_tags': all_ner_tags, 'country': all_regions,})
    return df

folder_path = r".\processed_annotated"
dataGlobal = load_tsvs_from_folder(folder_path)
dataGlobal['ner_tags'] = dataGlobal['ner_tags'].apply(lambda x: [LabelToID[i] for i in x])

In [43]:
def map_region(x):
    r1=['saud','nm','mauritius','ghana'] # Africa south
    r2=['kenya','et'] # Africa East
    r3=['eg','alg'] # Africa North
    r4=['af','panaf','afs'] # Africa General
    
    r5=['cosrica','cosric','cuba','elsalv','hond','mex','nic','pan'] # Central America
    r6=['canada','aus','nz'] # Ind. Reg
    
    r7=['oman','jr','israel','iran','uae','saud','qt','kw'] # Middle East

    r8=['in','pk','bangladesh'] # West Asia
    r9=['asia','cn','jp','kr','tw','mong'] # General Asia
    r10=['th','my'] # SEA Asia

    r11=['arg','bol','chile','col','ecuador','guyana','paraguay','peru','uruguay','ven'] # South America

    # Loop through regions, return number of region
    regions = [r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11]
    
    for i, region in enumerate(regions, start=1):
        if x in region:
            return i

In [44]:
dataGlobal['region']=dataGlobal['country'].apply(map_region)

In [45]:
dataGlobal

,tokens,ner_tags,country,region
0,"[DURBAN, ,, May, 20, 2022, (, IPS, ), -, Gover...","[5, 0, 1, 2, 2, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, ...",afs,4.0
1,"[These, were, among, the, diverse, opinions, o...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",afs,4.0
2,"[Hundreds, of, delegates, ,, including, world,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",afs,4.0
3,"[Sessions, and, panel, discussions, highlighte...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",afs,4.0
4,"[Speaking, during, the, closing, ceremony, on,...","[0, 0, 0, 0, 0, 0, 1, 0, 5, 6, 6, 6, 0, 0, 0, ...",afs,4.0
...,...,...,...,...
21312,"[Fanny, is, about, to, give, a, picture, of, h...","[3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",ven,11.0
21313,"[But, the, majority, of, the, civil, society, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",ven,11.0
21314,"[It, ’s, all, about, a, perpetual, and, useful...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, ...",ven,11.0
21315,"[Lusverti, adds, :, “, The, struggle, of, thes...","[3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",ven,11.0


In [46]:
dataGlobal.groupby('region').count()

,tokens,ner_tags,country
region,,,
1.0,1580,1580,1580
2.0,331,331,331
3.0,628,628,628
4.0,2222,2222,2222
5.0,2771,2771,2771
6.0,2503,2503,2503
7.0,2082,2082,2082
8.0,1969,1969,1969
9.0,3897,3897,3897


In [47]:
dataGlobal1=dataGlobal[dataGlobal['region']==1].copy()
dataGlobal1.sample(frac=1).reset_index(drop=True)

,tokens,ner_tags,country,region
0,"[And, it, ’s, not, about, failure, ,, it, ’s, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",nm,1.0
1,"[Historically, ,, many, farmers, with, smaller...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",nm,1.0
2,"[This, report, has, two, parts, .]","[0, 0, 0, 0, 0, 0]",mauritius,1.0
3,"[She, ,, however, ,, often, worries, about, wh...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",nm,1.0
4,"[Government, ,, pointed, out, the, VPM, ,, wil...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",mauritius,1.0
...,...,...,...,...
1575,"[“, We, need, to, have, more, competition, in,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",mauritius,1.0
1576,"[That, is, an, indication, that, the, governme...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",ghana,1.0
1577,"[She, has, seven, years, of, experience, in, t...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",nm,1.0
1578,"[The, statement, said, for, an, accused, perso...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",ghana,1.0
